# OpenPI pi0.5 xArm LoRA Fine-Tuning Pipeline

This notebook keeps raw robot data on Google Drive and uses Colab for OpenPI setup, LeRobot conversion, norm stats, GPU training, and final inference export.

Persistent Drive layout:

```text
MyDrive/embodied_ai_xarm/
  raw/                            # raw task/episode folders synced from the robot/local PC
    pick_up_blue_block/episode_*/
    place_red_on_blue/episode_*/
  code/                           # tiny project files copied from local when they change
    convert_xarm_raw_to_lerobot.py
    openpi_xarm_config.py
  lerobot/                        # persistent LeRobotDataset cache
  openpi_cache/pi05_base/params   # persistent pi05_base checkpoint
  openpi_assets/                  # persistent norm stats backup
  checkpoints/                    # persistent training outputs
  final_models/                   # inference exports
```


## Local Prep After Collecting More Data

When you add demonstrations, copy or sync raw task folders directly into Google Drive:

```text
MyDrive/embodied_ai_xarm/raw/<task>/episode_xxx/
```

Each episode should contain:

```text
meta.json
robot_log.csv
gripper_events.csv
realsense_0/
realsense_1/
realsense_2/
```

When converter/config changes, copy these two small files to Drive:

```text
MyDrive/embodied_ai_xarm/code/convert_xarm_raw_to_lerobot.py
MyDrive/embodied_ai_xarm/code/openpi_xarm_config.py
```

Raw data no longer goes into `xarm_colab_payload.zip`; Colab reads raw data directly from Drive.


## 1. Mount Drive and Configure Paths

Run this at the start of every Colab session. It defines persistent Drive paths and keeps uv/pip temp caches on Colab local disk.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os

DRIVE_ROOT = Path('/content/drive/MyDrive/embodied_ai_xarm')
RAW_ROOT = DRIVE_ROOT / 'raw'
CODE_ROOT = DRIVE_ROOT / 'code'
PROJECT_DIR = Path('/content/embodied-ai-xarm')
OPENPI_DIR = Path('/content/openpi')
HF_LEROBOT_HOME = DRIVE_ROOT / 'lerobot'
PI05_BASE_PARAMS = DRIVE_ROOT / 'openpi_cache/pi05_base/params'
ASSETS_BACKUP = DRIVE_ROOT / 'openpi_assets'
CHECKPOINT_BACKUP = DRIVE_ROOT / 'checkpoints'
FINAL_MODELS = DRIVE_ROOT / 'final_models'

# Google Drive is reliable for datasets/checkpoints, but not for uv/pip lock and temp files.
for p in [DRIVE_ROOT, RAW_ROOT, CODE_ROOT, HF_LEROBOT_HOME, PI05_BASE_PARAMS.parent, ASSETS_BACKUP, CHECKPOINT_BACKUP, FINAL_MODELS]:
    p.mkdir(parents=True, exist_ok=True)
Path('/content/uv_cache').mkdir(parents=True, exist_ok=True)
Path('/content/pip_cache').mkdir(parents=True, exist_ok=True)

os.environ['HF_LEROBOT_HOME'] = str(HF_LEROBOT_HOME)
os.environ['UV_CACHE_DIR'] = '/content/uv_cache'
os.environ['PIP_CACHE_DIR'] = '/content/pip_cache'
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.9'

print('Drive root:', DRIVE_ROOT)
print('Raw root:', RAW_ROOT, 'exists:', RAW_ROOT.exists())
print('Code root:', CODE_ROOT, 'files:', sorted(p.name for p in CODE_ROOT.glob('*')))
print('UV cache:', os.environ['UV_CACHE_DIR'])


## 2. Check GPU

A100 is preferred. L4 may work with small batch sizes. T4 is usually not enough for practical pi0.5 LoRA training.


In [ ]:
! nvidia-smi


## 3. Prepare Project Code and Check Drive Raw Data

Run this whenever you update `code/convert_xarm_raw_to_lerobot.py`, `code/openpi_xarm_config.py`, or add raw episodes under `raw/`. Raw data stays on Drive; this cell only copies the two small code files into Colab `/content`.


In [ ]:
from pathlib import Path
import shutil

converter_src = CODE_ROOT / 'convert_xarm_raw_to_lerobot.py'
config_src = CODE_ROOT / 'openpi_xarm_config.py'
assert converter_src.exists(), f'Missing {converter_src}. Copy the converter to Drive code/ first.'
assert config_src.exists(), f'Missing {config_src}. Copy openpi_xarm_config.py to Drive code/ first.'
assert RAW_ROOT.exists(), f'Missing raw root {RAW_ROOT}'

project_ft = PROJECT_DIR / 'fine_tune'
project_ft.mkdir(parents=True, exist_ok=True)
shutil.copy2(converter_src, project_ft / 'convert_xarm_raw_to_lerobot.py')
shutil.copy2(config_src, project_ft / 'openpi_xarm_config.py')

meta_files = sorted(RAW_ROOT.glob('*/*/meta.json'))
robot_logs = sorted(RAW_ROOT.glob('*/*/robot_log.csv'))
print('Copied code to:', project_ft)
print('Raw tasks:', sorted(p.name for p in RAW_ROOT.iterdir() if p.is_dir()))
print('Raw episodes with meta.json:', len(meta_files))
print('Raw episodes with robot_log.csv:', len(robot_logs))
assert meta_files, f'No meta.json files found under {RAW_ROOT}'
assert len(meta_files) == len(robot_logs), 'meta.json and robot_log.csv counts differ; check upload completeness.'

for task_dir in sorted(p for p in RAW_ROOT.iterdir() if p.is_dir()):
    eps = sorted(task_dir.glob('episode_*'))
    print(f'{task_dir.name}: {len(eps)} episodes')


In [ ]:
# Optional: inspect raw data directly on Drive.
!find "{RAW_ROOT}" -maxdepth 3 -type d -name "episode_*" | sort | head -20
!find "{RAW_ROOT}" -name meta.json | wc -l
!find "{RAW_ROOT}" -name robot_log.csv | wc -l


In [ ]:
# Raw data stays on Drive. No zip unpacking or local raw move is needed.
print('Project code dir:', PROJECT_DIR / 'fine_tune')
print('Drive raw root:', RAW_ROOT)


## 4. Clone and Install OpenPI

OpenPI lives in Colab `/content` for speed. Dependency caches stay on local Colab disk to avoid Google Drive lock-file issues.


In [ ]:
%%bash
set -e
cd /content
if [ ! -d openpi ]; then
  git clone --recurse-submodules https://github.com/Physical-Intelligence/openpi.git
fi
cd /content/openpi
pip install -q uv

# Temporarily move the cache to the local disk and disable hardlinks
export UV_CACHE_DIR=/tmp/uv_cache
export UV_LINK_MODE=copy

GIT_LFS_SKIP_SMUDGE=1 uv sync
GIT_LFS_SKIP_SMUDGE=1 uv pip install -e .


## 5. Patch OpenPI Config for xArm

This inserts `LeRobotXArmDataConfig` and the two xArm `TrainConfig`s. It also redirects the pi05 base weight loader to the Drive checkpoint cache.


In [ ]:
from pathlib import Path
import runpy

openpi_config = OPENPI_DIR / 'src/openpi/training/config.py'
snippet_file = PROJECT_DIR / 'fine_tune/openpi_xarm_config.py'
snippet = runpy.run_path(str(snippet_file))['XARM_CONFIG_SNIPPET']
marker = '# Add these TrainConfig entries to _CONFIGS.'
class_part, train_tail = snippet.split(marker, 1)
train_part = marker + train_tail

text = openpi_config.read_text()
if 'import openpi.policies.libero_policy as libero_policy' not in text:
    text = text.replace(
        'import openpi.policies.aloha_policy as aloha_policy\n',
        'import openpi.policies.aloha_policy as aloha_policy\nimport openpi.policies.libero_policy as libero_policy\n',
    )
if 'class LeRobotXArmDataConfig' not in text:
    text = text.replace('_CONFIGS = [', class_part.rstrip() + '\n\n_CONFIGS = [', 1)
if 'pi05_xarm_colab_smoke' not in text:
    text = text.replace('_CONFIGS = [', '_CONFIGS = [\n' + train_part.rstrip() + '\n', 1)

drive_weight_loader = f'weight_loader=weight_loaders.CheckpointWeightLoader("{PI05_BASE_PARAMS.as_posix()}")'
text = text.replace(
    'weight_loader=weight_loaders.CheckpointWeightLoader("gs://openpi-assets/checkpoints/pi05_base/params")',
    drive_weight_loader,
)
openpi_config.write_text(text)

!grep -n "LeRobotXArmDataConfig\|pi05_xarm_colab_smoke\|openpi_cache/pi05_base/params" /content/openpi/src/openpi/training/config.py


## 6. Convert Drive Raw Data to Persistent LeRobotDataset

Run this whenever raw data or the converter changes. It reads raw episodes directly from `MyDrive/embodied_ai_xarm/raw` and writes the real LeRobotDataset under Drive via `HF_LEROBOT_HOME`.


In [ ]:
%cd /content/openpi
!uv run python /content/embodied-ai-xarm/fine_tune/convert_xarm_raw_to_lerobot.py   --raw-root "{RAW_ROOT}"   --output-dir /content/embodied-ai-xarm/fine_tune/data/xarm_pi05_data/lerobot_light   --repo-id local/xarm_pi05_data   --robot-type xarm6   --fps 10   --overwrite
!find /content/drive/MyDrive/embodied_ai_xarm/lerobot/local/xarm_pi05_data -maxdepth 3 -type f | head -30
!cat /content/drive/MyDrive/embodied_ai_xarm/lerobot/local/xarm_pi05_data/meta/info.json


## 7. Cache pi05 Base Checkpoint on Drive

Run this only if `openpi_cache/pi05_base/params` is missing or incomplete. Once cached, training restores from Drive instead of repeatedly downloading from GCS.


In [ ]:
if not (PI05_BASE_PARAMS / 'commit_success.txt').exists():
    print('Caching pi05_base params to Drive. This may take a while.')
    !pip uninstall -y crcmod >/dev/null 2>&1 || true
    !pip install --no-cache-dir -U crcmod
    !python -c "import crcmod._crcfunext; print('compiled crcmod OK')" || printf "[GSUtil]\\ncheck_hashes = if_fast_else_skip\\n" > ~/.boto
    !rm -rf "{PI05_BASE_PARAMS}" "{PI05_BASE_PARAMS}.partial"
    !mkdir -p "{PI05_BASE_PARAMS}"
    !gsutil -m cp -r gs://openpi-assets/checkpoints/pi05_base/params/* "{PI05_BASE_PARAMS}/"
else:
    print('pi05_base params already cached:', PI05_BASE_PARAMS)
!ls -lah "{PI05_BASE_PARAMS}" | head


## 8. Compute or Restore Norm Stats

Run this when raw data, action definitions, or the converter changes. The stats are copied to Drive so future sessions can restore them without recomputing. If you add data, keep `RECOMPUTE_NORM_STATS = True`.


In [ ]:
RECOMPUTE_NORM_STATS = True  # Set False only when raw data and converter did not change.

%cd /content/openpi
DATASET_INFO = HF_LEROBOT_HOME / 'local/xarm_pi05_data/meta/info.json'
assert DATASET_INFO.exists(), f'Missing LeRobotDataset metadata: {DATASET_INFO}. Run Step 6 first.'

backup = ASSETS_BACKUP / 'pi05_xarm_colab_smoke'
local_assets = OPENPI_DIR / 'assets/pi05_xarm_colab_smoke'

if backup.exists() and not RECOMPUTE_NORM_STATS:
    !mkdir -p /content/openpi/assets
    !rm -rf "{local_assets}"
    !cp -r "{backup}" /content/openpi/assets/
    print('Restored norm stats from Drive:', backup)
else:
    !rm -rf "{local_assets}" "{backup}"
    !uv run scripts/compute_norm_stats.py --config-name pi05_xarm_colab_smoke
    !mkdir -p "{ASSETS_BACKUP}"
    !cp -r "{local_assets}" "{ASSETS_BACKUP}/"
    print('Computed and backed up norm stats:', backup)


## 9. Smoke Train

This runs the 1,000-step LoRA smoke test. Use it after pipeline changes before launching a longer run.


In [ ]:
%cd /content/openpi
!uv run scripts/train.py pi05_xarm_colab_smoke --exp-name=xarm_test --overwrite


## 10. Back Up Smoke Checkpoint

Run this immediately after training completes. Colab `/content` is temporary; Drive is persistent.


In [ ]:
!mkdir -p /content/drive/MyDrive/embodied_ai_xarm/checkpoints
!cp -r /content/openpi/checkpoints/pi05_xarm_colab_smoke /content/drive/MyDrive/embodied_ai_xarm/checkpoints/
!find /content/drive/MyDrive/embodied_ai_xarm/checkpoints/pi05_xarm_colab_smoke -maxdepth 3 -type d | sort | tail -20


## 11. Longer Training

Use this only after the smoke run finishes cleanly. For your current small dataset, consider editing `pi05_xarm` to 5,000 steps first, then scale up.


In [ ]:
%cd /content/openpi
uv run scripts/train.py pi05_xarm \
  --exp-name=xarm_pi05_lora \
  --checkpoint-dir=/content/drive/MyDrive/embodied_ai_xarm/checkpoints


In [ ]:
!uv run scripts/train.py pi05_xarm --exp-name=xarm_pi05_lora --overwrite


## 12. Save the model to Google Drive


In [ ]:
%%bash
set -e

SRC_ROOT=/content/openpi/checkpoints/pi05_xarm/xarm_pi05_lora
FINAL_ROOT=/content/drive/MyDrive/embodied_ai_xarm/final_models/pi05_xarm_xarm_pi05_lora

LATEST=$(find "$SRC_ROOT" -maxdepth 1 -type d -regex '.*/[0-9]+' | sed 's#.*/##' | sort -n | tail -1)

if [ -z "$LATEST" ]; then
  echo "No checkpoint found under $SRC_ROOT"
  exit 1
fi

echo "Saving latest checkpoint for inference: $LATEST"

mkdir -p "$FINAL_ROOT/$LATEST"
rsync -ah --delete --info=progress2 "$SRC_ROOT/$LATEST/" "$FINAL_ROOT/$LATEST/"

cat > "$FINAL_ROOT/README.txt" <<EOF
OpenPI xArm pi0.5 LoRA fine-tuned checkpoint.

Config name:
pi05_xarm

Checkpoint step:
$LATEST

Use for inference:
uv run scripts/serve_policy.py policy:checkpoint \\
  --policy.config=pi05_xarm \\
  --policy.dir=$FINAL_ROOT/$LATEST

Expected observation keys:
image, wrist_image, state, prompt

State/action meaning:
[j1_rad, j2_rad, j3_rad, j4_rad, j5_rad, j6_rad, gripper_mm]
EOF

echo "Saved final inference model to:"
echo "$FINAL_ROOT/$LATEST"

echo "Preview:"
find "$FINAL_ROOT/$LATEST" -maxdepth 2 -type d | sort


In [ ]:
!find /content/drive/MyDrive/embodied_ai_xarm/final_models/pi05_xarm_xarm_pi05_lora/29999 -maxdepth 3 -type f | head -30


In [ ]:
!find /content/drive/MyDrive/embodied_ai_xarm -maxdepth 6 -type d -name "29999" 2>/dev/null


In [ ]:
!rm -rf /content/tmp_pi05_xarm_29999/29999/train_state


In [ ]:
%%bash
set -e

MODEL_ROOT=/content/drive/MyDrive/embodied_ai_xarm/final_models/pi05_xarm_xarm_pi05_lora
TMP=/content/tmp_pi05_xarm_export
OUT_DIR=/content/drive/MyDrive/embodied_ai_xarm

STEP=$(find "$MODEL_ROOT" -maxdepth 1 -type d -regex '.*/[0-9]+' 2>/dev/null | sed 's#.*/##' | sort -n | tail -1)

if [ -z "$STEP" ]; then
  echo "No numeric checkpoint step found under:"
  echo "$MODEL_ROOT"
  echo "Searching wider..."
  find /content/drive/MyDrive/embodied_ai_xarm -maxdepth 8 -type d -regex '.*/[0-9]+' 2>/dev/null | sort
  exit 1
fi

SRC="$MODEL_ROOT/$STEP"
OUT="$OUT_DIR/pi05_xarm_${STEP}_inference.tar.gz"

echo "Using checkpoint step: $STEP"
echo "Source: $SRC"
echo "Output: $OUT"

echo "Source structure:"
find "$SRC" -maxdepth 2 -type d | sort

rm -rf "$TMP"
mkdir -p "$TMP/$STEP"

echo "Copying checkpoint to local tmp..."
rsync -ah --info=progress2 "$SRC/" "$TMP/$STEP/"

echo "Removing train_state for inference-only export..."
rm -rf "$TMP/$STEP/train_state"

echo "Verifying inference package contents..."
test -d "$TMP/$STEP/assets"
test -d "$TMP/$STEP/params"
find "$TMP/$STEP" -maxdepth 2 -type d | sort

echo "Creating tar.gz locally..."
tar -czf "/content/pi05_xarm_${STEP}_inference.tar.gz" -C "$TMP" "$STEP"

echo "Copying tar.gz back to Drive..."
cp "/content/pi05_xarm_${STEP}_inference.tar.gz" "$OUT"

echo "Done."
ls -lh "$OUT"


In [ ]:
! find /content/drive/MyDrive/embodied_ai_xarm/final_models/pi05_xarm_xarm_pi05_lora/29999/assets -name "norm_stats.json" -print
